# Step 6 — Classify bot vs. human authors

> Append `is_bot` columns derived from author/login fields. Recorded for descriptive reporting only (tallied in 08b's `bot_vs_human`); step 07's IC/EC logic never reads `is_bot`.

**Paper section:** Step 1 — Repository selection and workflow extraction (offline preparation).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/5_dataset.csv` | CSV | Step 5 |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/6_dataset.csv` | CSV (with bot-classification columns) | Step 7 (offline IC/EC) |
| `data/step6_report.json` | JSON | Step 7 |




In [2]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 6 — starting: classify authors as bot or human...", flush=True)
"""Step 6: classify bot vs human authors.

Adds an `is_bot` column to the dataset based on author_name patterns.
Preserves all rows — researchers can filter as needed.
"""

import argparse
import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path

import polars as pl

BOT_PATTERNS = [
    re.compile(r"\[bot\]$"),
    re.compile(r"\b[Bb]ot\b"),
    re.compile(r"\b[Rr]obot\b"),
    re.compile(r"\b[Rr]enovate\b"),
    re.compile(r"\b[Dd]ependabot\b"),
    re.compile(r"\b[Cc]opilot\b"),
    re.compile(r"\b[Mm]ergif"),
    re.compile(r"\b[Pp]row"),
    re.compile(r"\b[Cc]opybara"),
    re.compile(r"\b[Aa]utomation\b"),
    re.compile(r"\b[Ss]teward\b"),
    re.compile(r"\b[Bb]ors\b"),
    re.compile(r"\b[Cc][Ii]\b"),
    re.compile(r"\b[Aa]ction[s]?\b"),
    re.compile(r"\b[Dd]ep[fu]"),
    re.compile(r"\b[Oo]wl\s*[Bb]ot\b"),
    re.compile(r"\b[Ss]ecurity\s*[Bb]ot\b"),
    re.compile(r"\b[Rr]elease\s*[Bb]ot\b"),
    re.compile(r"\b[Tt]ranslation\s*[Bb]ot\b"),
    re.compile(r"\b[Dd]iscoursebot\b"),
    re.compile(r"\b[Rr]iot[Rr]obot\b"),
    re.compile(r"\b[Jj]unie\b"),
    re.compile(r"\b[Jj]arvis"),
    re.compile(r"\b[Ss]peakeasy\b"),
    re.compile(r"\b[Tt]ag[Bb]ot\b"),
    re.compile(r"\b[Ff]ern\b"),
    re.compile(r"\b[Pp]rojen\b"),
    re.compile(r"\b[Zz]uul\b"),
    re.compile(r"\b[Qq]odana\b"),
    re.compile(r"\b[Nn]autobot\b"),
    re.compile(r"\b[Rr]obin\b"),
]

FALSE_POSITIVE_NAMES = {
    "Peter Sobot",
    "Maxwell Talbot",
    "Gabriel Poulenard-Talbot",
    "Lorenzo Zabot",
    "Michał Chrobot",
    "michalChrobot",
    "Darrin Talbot",
    "mjtalbot",
    "Timid Robot Zehta",
    "Robin",
}

def is_bot_author(name: str | None) -> bool:
    if name is None:
        return False
    if name in FALSE_POSITIVE_NAMES:
        return False
    return any(p.search(name) for p in BOT_PATTERNS)

@dataclass
class ClassificationStats:
    total_rows: int = 0
    bot_rows: int = 0
    human_rows: int = 0
    unique_authors: int = 0
    unique_bots: int = 0
    unique_humans: int = 0

def classify_authors(input_path: Path, output_path: Path) -> ClassificationStats:
    stats = ClassificationStats()

    from dataset_utils.integrity import assert_input_matches_report

    # Assert upstream integrity BEFORE overwriting (step 6 output may equal input).
    input_sha = assert_input_matches_report(
        input_path=input_path,
        prior_report_path=Path("data/step5_report.json"),
        label="step 6 input",
    )

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(input_path, encoding="utf-8-sig")
    stats.total_rows = len(df)

    df = df.with_columns(
        pl.col("author_name").map_elements(is_bot_author, return_dtype=pl.Boolean, skip_nulls=False).alias("is_bot")
    )

    stats.bot_rows = df.filter(pl.col("is_bot") == True).height
    stats.human_rows = df.filter(pl.col("is_bot") == False).height

    unique_authors_df = df.select("author_name", "is_bot").unique()
    stats.unique_authors = len(unique_authors_df)
    stats.unique_bots = unique_authors_df.filter(pl.col("is_bot") == True).height
    stats.unique_humans = unique_authors_df.filter(pl.col("is_bot") == False).height

    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("  writing output...", flush=True)
    df.write_csv(output_path, include_header=True)

    from dataset_utils.hashing import compute_file_hash
    report = {
        **asdict(stats),
        "input_sha256": input_sha,
        "output_file": str(output_path),
        "output_sha256": compute_file_hash(output_path),
    }
    report_path = output_path.parent / "step6_report.json"
    with report_path.open("w") as f:
        json.dump(report, f, indent=2)

    return stats

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Classify bot vs human authors in the dataset.")
    parser.add_argument(
        "--input",
        type=Path,
        default=Path("data/5_dataset.csv"),
        help="Input dataset CSV file path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/6_dataset.csv"),
        help="Output dataset CSV with is_bot column.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    stats = classify_authors(args.input, args.output)

    print("Author classification finished.")
    print(f"Input: {args.input}")
    print(f"Output: {args.output}")
    print("-" * 40)
    print(f"Total rows: {stats.total_rows}")
    print(f"Bot rows: {stats.bot_rows} ({stats.bot_rows / stats.total_rows * 100:.1f}%)")
    print(f"Human rows: {stats.human_rows} ({stats.human_rows / stats.total_rows * 100:.1f}%)")
    print("-" * 40)
    print(f"Unique authors: {stats.unique_authors}")
    print(f"Unique bots: {stats.unique_bots}")
    print(f"Unique humans: {stats.unique_humans}")

main()


Step 6 — starting: classify authors as bot or human...


  reading input CSV (may take a moment for large files)...
  writing output...
Author classification finished.
Input: data/5_dataset.csv
Output: data/6_dataset.csv
----------------------------------------
Total rows: 253921
Bot rows: 33895 (13.3%)
Human rows: 220026 (86.7%)
----------------------------------------
Unique authors: 58110
Unique bots: 455
Unique humans: 57655


---

**Next step:** [`07_offline_apply_criteria.ipynb`](./07_offline_apply_criteria.ipynb)
